# Concatenación y homogeneización de bases de isótopos foliares (v2)

Combina las **5 bases** en una sola tabla homogénea:

| Base | Archivo | Sitios |
|---|---|---|
| Galeras | `gal_complete.xlsx` | Galeras |
| Selva Viva | `SV_complete.xlsx` (hoja `todo`) | Selva Viva |
| Sumaco | `Sumaco_complete.xlsx` | Sumaco |
| iWUE | `iWUE_OYC_GUA.xlsx` | Oyacachi, Guacamayos |
| **NUMEXC (nuevo)** | `NUMEXC.xlsx` | BoNu, SFNu, CaNu |

**Qué hace, en orden:**
1. Carga cada archivo y **mapea sus columnas a un esquema común** (cada base trae nombres distintos; p. ej. en Sumaco el C bulk se llama `[C]` y el δ15N `δ15N (‰ v.s.AIR)`).
2. Limpia `Plot`, `Sample_ID`, `old_treeID`, `new_TreeID`.
3. **Homogeneiza unidades**: todas las concentraciones ([C] celulosa, [C] bulk, [N], [O]) quedan en **%**. Las que venían como fracción (0–1) se multiplican por 100.
4. Aplica las correcciones manuales ya verificadas (parcelas, intercambio de columnas en Oyacachi).
5. Concatena, añade elevación y una llave única de árbol (`Tree_key`) que une el ID viejo y el nuevo.
6. Control de calidad (`QC_flag`).
7. **Pendientes**: qué muestras aún no tienen celulosa / bulk medido.
8. **Coincidencias**: cuántos árboles tienen muestras en más de un año, por sitio, y cuántos años los separan.
9. Exporta un Excel con **las celdas vacías resaltadas**.

## 1. Librerías y rutas

Solo cambia `BASE_DIR`. Los archivos se buscan **por nombre** dentro de `BASE_DIR` (en cualquier subcarpeta), así que no importa si `NUMEXC.xlsx` está en su propia carpeta o en la raíz.

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

CANDIDATOS = [
    Path(r"C:\Users\selene.baez\Downloads\NAPO_preliminar"),   # PC trabajo
    Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar"),        # PC personal
    Path.cwd(),                                                # carpeta actual (respaldo)
]
BASE_DIR = next(p for p in CANDIDATOS if p.exists())
print('BASE_DIR =', BASE_DIR)

ARCHIVOS = {
    'Galeras':    'gal_complete.xlsx',
    'Selva Viva': 'SV_complete.xlsx',
    'Sumaco':     'Sumaco_complete.xlsx',
    'iWUE':       'iWUE_OYC_GUA.xlsx',
    'NUMEXC':     'NUMEXC.xlsx',
}

def buscar(nombre):
    """Devuelve la ruta del archivo buscándolo por nombre dentro de BASE_DIR."""
    directo = BASE_DIR / nombre
    if directo.exists():
        return directo
    hits = sorted(BASE_DIR.rglob(nombre))
    if not hits:
        raise FileNotFoundError(f'No encuentro {nombre} dentro de {BASE_DIR}')
    return hits[0]

RUTAS = {k: buscar(v) for k, v in ARCHIVOS.items()}
for k, v in RUTAS.items():
    print(f'{k:11s} -> {v}')

BASE_DIR = /home/claude/data
Galeras     -> /home/claude/data/gal_complete.xlsx
Selva Viva  -> /home/claude/data/SV_complete.xlsx
Sumaco      -> /home/claude/data/Sumaco_complete.xlsx
iWUE        -> /home/claude/data/iWUE_OYC_GUA.xlsx
NUMEXC      -> /home/claude/data/NUMEXC.xlsx


## 2. Diccionario de elevación por parcela

NUMEXC no necesita diccionario: el archivo ya trae `elevation_m` por sitio (BoNu 1000, SFNu 2000, CaNu 3000 m).

In [2]:
elevation_dict = {
    # Galeras
    ("Galeras", 20): 1090, ("Galeras", 21): 1080, ("Galeras", 22): 1060,
    ("Galeras", 23): 1090, ("Galeras", 24): 1110, ("Galeras", 29): 1050,
    ("Galeras", 47): 1000, ("Galeras", 48): 1130, ("Galeras", 49): 1080,
    ("Galeras", 28): 1597,
    ("Galeras", 25): 1450, ("Galeras", 26): 1560, ("Galeras", 50): 1557,
    ("Galeras", 60): 1570, ("Galeras", 61): 1560, ("Galeras", 62): 1590,
    # Oyacachi
    ("Oyacachi", 81): 4000, ("Oyacachi", 82): 4000, ("Oyacachi", 83): 4000,
    ("Oyacachi", 84): 3500, ("Oyacachi", 85): 3500, ("Oyacachi", 86): 3500,
    # Sumaco
    ("Sumaco", 9): 2000, ("Sumaco", 10): 2000, ("Sumaco", 11): 2000,
    ("Sumaco", 12): 2000, ("Sumaco", 13): 2000,
    ("Sumaco", 16): 1500, ("Sumaco", 18): 1500,
    # Guacamayos
    ("Guacamayos", 6): 2000, ("Guacamayos", 7): 1940, ("Guacamayos", 8): 1995,
    ("Guacamayos", 55): 2000, ("Guacamayos", 27): 1990,
    # Selva Viva
    ("Selva Viva", 68): 502, ("Selva Viva", 69): 508, ("Selva Viva", 70): 508,
    ("Selva Viva", 72): 453, ("Selva Viva", 74): 470,
}

## 3. Esquema común de columnas y funciones auxiliares

**Nombres nuevos de las columnas de isótopos** (las unidades van en el nombre):

| Columna nueva | Qué es | Nombre(s) original(es) |
|---|---|---|
| `C_cel_pct` | % C en celulosa | `[C]%` |
| `d13C_cel` | δ13C celulosa (‰ VPDB) | `δ13C (‰ v.s.V-PDB)` |
| `C_bulk_pct` | % C en hoja (bulk) | `[C_b]%`, Sumaco: `[C]` |
| `d13C_bulk` | δ13C bulk (‰ VPDB) | `bulk_δ13C (‰ v.s.V-PDB)` |
| `N_bulk_pct` | % N en hoja (bulk) | `[N]` |
| `d15N_bulk` | δ15N bulk (‰ **AIR**) | `δ15N (‰ v.s. V-PDB)`, Sumaco: `δ15N (‰ v.s.AIR)` |
| `O_pct` | % O (solo NUMEXC) | `[O]` |
| `d18O` | δ18O (‰ VSMOW, solo NUMEXC) | `δ18O in ‰ vs DeltaSMOW` |

Nota: el δ15N siempre se reporta contra N₂ atmosférico (AIR); la etiqueta "V-PDB" de las planillas es un error de nombre, no de dato.

In [3]:
ISO_MAP = {
    '[C]%': 'C_cel_pct',
    'δ13C (‰ v.s.V-PDB)': 'd13C_cel',
    '[C_b]%': 'C_bulk_pct',
    'bulk_δ13C (‰ v.s.V-PDB)': 'd13C_bulk',
    '[N]': 'N_bulk_pct',
    'δ15N (‰ v.s. V-PDB)': 'd15N_bulk',
    'δ15N (‰ v.s.AIR)': 'd15N_bulk',
    '[O]': 'O_pct',
    'δ18O in ‰ vs DeltaSMOW': 'd18O',
}
CORE_ISO = ['C_cel_pct', 'd13C_cel', 'C_bulk_pct', 'd13C_bulk', 'N_bulk_pct', 'd15N_bulk']
O_ISO = ['O_pct', 'd18O']
PCT_COLS = ['C_cel_pct', 'C_bulk_pct', 'N_bulk_pct', 'O_pct']
CEL = ['C_cel_pct', 'd13C_cel']
BULK = ['C_bulk_pct', 'd13C_bulk', 'N_bulk_pct', 'd15N_bulk']

def clean_plot(v):
    """'p09', ' 24 ', '9.0', 9 -> 9 (entero)."""
    if pd.isna(v):
        return pd.NA
    s = str(v).strip().lower()
    s = re.sub(r'\.0$', '', s).lstrip('p').lstrip('0') or '0'
    return int(s) if s.isdigit() else pd.NA

def to_id(v):
    """Extrae el número de un ID de árbol ('1461 (silica...)' -> 1461; 'no ID', 'dead' -> NA)."""
    if pd.isna(v):
        return pd.NA
    m = re.match(r'^\s*(\d+)', str(v))
    return int(m.group(1)) if m else pd.NA

def clean_sample_id(v):
    if pd.isna(v):
        return pd.NA
    return re.sub(r'\.0$', '', str(v).strip())

def implied_plot(sample_id):
    """'p09-230' -> 9 ; 'jh-p60-3657' -> 60."""
    if pd.isna(sample_id):
        return pd.NA
    m = re.match(r'^(?:jh-)?p0*(\d+)-', str(sample_id).strip().lower())
    return int(m.group(1)) if m else pd.NA

def id_number(sample_id):
    """Número final del Sample_ID ('p28-8726' -> 8726)."""
    if pd.isna(sample_id):
        return pd.NA
    m = re.search(r'-0*(\d+)$', str(sample_id))
    return int(m.group(1)) if m else pd.NA

conversiones = []   # registro de cuántos valores se pasaron de fracción a %

def a_porcentaje(df, base):
    """Fracción (0–1) -> % (x100). Valores con |x| >= 1 ya están en %.
    Es seguro porque en %: C ~ 35–55, N ~ 0.5–4; en fracción: C ~ 0.35–0.55, N ~ 0.005–0.05
    (el rango de N en fracción nunca llega a 0.1, así que no hay solapamiento)."""
    for c in PCT_COLS:
        if c not in df:
            continue
        mask = df[c].notna() & (df[c].abs() < 1)
        conversiones.append({'base': base, 'columna': c,
                             'n_valores': int(df[c].notna().sum()),
                             'n_convertidos_x100': int(mask.sum())})
        df.loc[mask, c] = df.loc[mask, c] * 100
    return df

TYPE_MAP = {'herbarium': 'leaf herbarium'}

def estandarizar(df, base):
    """Renombra columnas de isótopos y básicas, limpia IDs y unidades."""
    df = df.drop(columns=[c for c in df.columns if str(c).startswith('Unnamed')], errors='ignore')
    df = df.rename(columns=ISO_MAP).rename(columns={'Sample-ID': 'Sample_ID'})
    for c in CORE_ISO + O_ISO:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df['Plot'] = df['Plot'].apply(clean_plot).astype('Int64')
    df['Sample_ID'] = df['Sample_ID'].apply(clean_sample_id)
    for c in ['old_treeID', 'new_TreeID']:
        if c in df:
            df[c + '_raw'] = df[c].astype('string')   # se conserva el texto original
            df[c] = df[c].apply(to_id).astype('Int64')
    if 'Type' in df:
        df['Type'] = df['Type'].replace(TYPE_MAP)
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
    df['source_file'] = ARCHIVOS[base]
    df['source_row'] = np.arange(len(df)) + 2      # fila en el Excel original (1 = encabezado)
    return a_porcentaje(df, base)

## 4. Cargar y estandarizar cada base

### 4.1 Galeras
Las filas `jh-...` sin año son vouchers de herbario de la campaña 2006 (mismo formato que los demás `jh-` de 2006) → se imputa `Year = 2006`. La fila `p23-8987` sin año usa un tag nuevo (89xx) → `Year = 2025`. Ambas imputaciones quedan marcadas en `Year_imputado`.

In [4]:
g = pd.read_excel(RUTAS['Galeras'])
g['Site'] = 'Galeras'
g['Project'] = 'NAPO'
g = estandarizar(g, 'Galeras')

g['Year_imputado'] = False
m_jh = g['Year'].isna() & g['Sample_ID'].str.startswith('jh-', na=False)
g.loc[m_jh, ['Year', 'Year_imputado']] = [2006, True]
m_new = g['Year'].isna() & (g['Sample_ID'].apply(id_number).fillna(0) >= 8000)
g.loc[m_new, ['Year', 'Year_imputado']] = [2025, True]
print('Años imputados:', g['Year_imputado'].sum())
g.head(2)

Años imputados: 56


,Sample_ID,C_cel_pct,d13C_cel,Type,N_bulk_pct,C_bulk_pct,d15N_bulk,d13C_bulk,Year,old_treeID,description,Type 2,Plot,new_TreeID,family,JH herbarium collections,genus,species,numeroCampo,Site,Project,old_treeID_raw,new_TreeID_raw,source_file,source_row,Year_imputado
0,p23-557,45.552995,-29.289590,leaf pulverized,1.912373,48.553106,1.960678,-31.458731,2006,557,NaN,NaN,23,<NA>,Quiinaceae,3737,Lacunaria,crenata,NaN,Galeras,NAPO,557,<NA>,gal_complete.xlsx,2,False
1,p23-557,43.360479,-29.427788,leaf pulverized,1.912373,48.553106,1.960678,-31.458731,2006,557,NaN,NaN,23,<NA>,Quiinaceae,3737,Lacunaria,crenata,NaN,Galeras,NAPO,557,<NA>,gal_complete.xlsx,3,False


### 4.2 Selva Viva
Se usa la hoja `todo` (incluye `pendientes`). Las columnas `falta_*` y `pendiente` se recalculan más abajo para **todas** las bases, así que se descartan aquí.

In [5]:
sv = pd.read_excel(RUTAS['Selva Viva'], sheet_name='todo')
sv['Site'] = 'Selva Viva'
sv['Project'] = 'NAPO'
sv = sv.drop(columns=['sample-ID', '_merge', 'pendiente'] +
             [c for c in sv.columns if c.startswith('falta_')], errors='ignore')
sv = estandarizar(sv, 'Selva Viva')
sv['JH herbarium collections'] = sv['JH herbarium collections'].astype('string')
sv.head(2)

,old_treeID,new_TreeID,family,genus,species,JH herbarium collections,Plot,Year,numeroCampo,description,C_cel_pct,d13C_cel,N_bulk_pct,C_bulk_pct,d15N_bulk,d13C_bulk,Sample_ID,Type,Site,Project,old_treeID_raw,new_TreeID_raw,source_file,source_row
0,2133,8876,Sapotaceae,Pouteria,torta,3918,68,2006,3918.0,plot 68 #2133,45.42,-31.66,2.353848,47.802782,-1.073430,-33.541062,jh-p68-3918,leaf herbarium,Selva Viva,NAPO,2133.0,8876.0,SV_complete.xlsx,2
1,2136,8878,Rubiaceae,Duroia,hirsuta,3920,68,2006,3920.0,plot 68 #2136,44.32,-32.58,1.898178,44.875144,1.036119,-34.404433,jh-p68-3920,leaf herbarium,Selva Viva,NAPO,2136.0,8878.0,SV_complete.xlsx,3


### 4.3 Sumaco
**Corrección importante respecto a la v1:** este archivo sí trae datos bulk, con otros nombres: `[C]` = C bulk (fracción), `[N]` = N bulk (fracción) y `δ15N (‰ v.s.AIR)`. En la v1 quedaban en columnas separadas y el bulk de Sumaco aparecía vacío.

La última fila no tiene `Sample-ID` pero sí `Sample ID` (`p12-221`) → se rellena desde ahí.

In [6]:
su = pd.read_excel(RUTAS['Sumaco'])
su['Site'] = 'Sumaco'
su['Project'] = 'NAPO'
su = su.rename(columns={'[C]': 'C_bulk_pct', 'Date analysis bulk': 'Date_analysis_bulk'})
su['Sample-ID'] = su['Sample-ID'].fillna(su['Sample ID'])
su = su.drop(columns=['Sample ID'])
su = estandarizar(su, 'Sumaco')
su.head(2)

,Sample_ID,family,genus,species,C_cel_pct,d13C_cel,Plot,Year,old_treeID,new_TreeID,JH herbarium collections,Type,N_bulk_pct,C_bulk_pct,d15N_bulk,d13C_bulk,Date_analysis_bulk,Site,Project,old_treeID_raw,new_TreeID_raw,source_file,source_row
0,p09-225,Moraceae,Ficus,quijosana,42.578188,-28.691043,9,2006,225,<NA>,NaN,leaf pulverized,1.890588,47.475213,0.217932,-31.520038,28092026.0,Sumaco,NAPO,225,<NA>,Sumaco_complete.xlsx,2
1,p09-230,Lauraceae,Ocotea,insularis,45.456318,-29.289822,1,2006,230,<NA>,3343.0,leaf pulverized,2.931468,44.982014,1.425977,-30.258210,28092026.0,Sumaco,NAPO,230,<NA>,Sumaco_complete.xlsx,3


### 4.4 iWUE (Oyacachi / Guacamayos)

In [7]:
iw = pd.read_excel(RUTAS['iWUE'])
iw['Project'] = 'NAPO'
iw = iw.rename(columns={'species2': 'species', 'herbarium specimen': 'JH herbarium collections'})
# [C]%_new es duplicado de [C]% donde existe; se usa solo para rellenar huecos
iw['[C]%'] = iw['[C]%'].fillna(iw['[C]%_new'])
iw = iw.drop(columns=['[C]%_new', 'species3', 'species4', 'species5', 'genus_species'], errors='ignore')
iw = estandarizar(iw, 'iWUE')
iw['JH herbarium collections'] = iw['JH herbarium collections'].astype('Int64').astype('string')
iw.head(2)

,new_TreeID,old_treeID,Plot,Site,family,leaf_sample_ID_2011,JH herbarium collections,C_cel_pct,d13C_cel,N_bulk_pct,d15N_bulk,C_bulk_pct,d13C_bulk,Year,Type,Sample_ID,genus,species,comment,Project,old_treeID_raw,new_TreeID_raw,source_file,source_row
0,4039,4039,82,Oyacachi,Solanaceae,56077.0,4194,45.059988,-24.65,1.518827,-2.154079,44.042720,-27.419388,2011,leaf pulverized,p82-4039,Solanum,stenophyllum,NaN,NAPO,4039.0,4039.0,iWUE_OYC_GUA.xlsx,2
1,8016,4020,81,Oyacachi,Rosaceae,56069.0,<NA>,45.781855,-26.37,2.984893,0.134936,45.888802,-28.960106,2011,leaf pulverized,p81-4020,Polylepis,pauta,NaN,NAPO,4020.0,8016.0,iWUE_OYC_GUA.xlsx,3


### 4.5 NUMEXC (nuevo)

- Sitios: `BoNu`, `SFNu`, `CaNu` (experimento NUMEX). `Sub = C` → subparcela control.
- Las filas 2023/2024 no traen `Year` ni `Sample-ID`: el año se toma de `Sampling date` y se construye un `Sample_ID` = `NX-<Sitio>-p<Plot>-<TreeID>`.
- El ID de árbol es `TreeID` (no hay ID viejo/nuevo).
- Las columnas de rasgos (LMA, nutrientes, iWUE, etc.) no se meten en la tabla de isótopos; quedan en la hoja `NUMEXC_rasgos` del Excel de salida.
- La fila `p09-01343` no tiene sitio, parcela ni árbol → se marca para revisión.

In [8]:
nx_raw = pd.read_excel(RUTAS['NUMEXC'])
nx = nx_raw.copy()
nx['Project'] = 'NUMEX'
nx['Year'] = nx['Year'].fillna(nx['Sampling date'].dt.year)
nx = nx.rename(columns={'TreeID': 'new_TreeID', 'specie': 'species', 'Sub': 'Subplot',
                        'elevation_m': 'Elevation_m', 'Sampling date': 'Sampling_date'})
nx['Plot'] = nx['Plot'].apply(clean_plot)
sid = 'NX-' + nx['Site'].astype('string') + '-p' + nx['Plot'].astype('Int64').astype('string').str.zfill(2) \
      + '-' + nx['new_TreeID'].astype('Int64').astype('string')
nx['Sample_ID'] = nx['Sample-ID'].astype('string').fillna(sid)
nx['Lab_ID'] = nx['Sample-ID']
nx['Type'] = 'leaf pulverized'
keep = ['Project', 'Site', 'Plot', 'Subplot', 'new_TreeID', 'Sample_ID', 'Lab_ID', 'Year', 'Sampling_date',
        'Type', 'genus', 'species', 'Elevation_m', 'Batch', 'plot/tree'] + list(ISO_MAP)
nx = nx[[c for c in keep if c in nx.columns]].drop(columns=['Sample-ID'], errors='ignore')
nx = estandarizar(nx, 'NUMEXC')
nx.head(3)

,Project,Site,Plot,Subplot,new_TreeID,Sample_ID,Lab_ID,Year,Sampling_date,Type,genus,species,Elevation_m,Batch,plot/tree,C_cel_pct,d13C_cel,C_bulk_pct,d13C_bulk,N_bulk_pct,d15N_bulk,O_pct,d18O,new_TreeID_raw,source_file,source_row
0,NUMEX,CaNu,1,C,1608,NX-CaNu-p01-1608,NaN,2024,2024-12-04,leaf pulverized,Weinmania,loxensis,3000.0,NaN,NaN,42.68,-27.67,44.801916,-29.445403,1.131803,-2.576851,45.610999,25.879051,1608.0,NUMEXC.xlsx,2
1,NUMEX,CaNu,1,C,5861,NX-CaNu-p01-5861,NaN,2024,2024-12-04,leaf pulverized,Weinmania,loxensis,3000.0,NaN,NaN,43.50,-28.09,45.356427,-29.504389,1.067421,-5.350569,45.031794,25.932011,5861.0,NUMEXC.xlsx,3
2,NUMEX,BoNu,4,C,2048,NX-BoNu-p04-2048,NaN,2024,2024-11-18,leaf pulverized,Pouteria,torta,1000.0,NaN,NaN,39.77,-30.98,48.386649,-33.402050,1.658410,1.888274,49.638377,31.032148,2048.0,NUMEXC.xlsx,4


## 5. Concatenar y aplicar correcciones manuales ya verificadas

In [9]:
all_df = pd.concat([g, sv, su, iw, nx], ignore_index=True, sort=False)
print('Filas totales:', len(all_df))
print(all_df.groupby(['Project', 'Site'], dropna=False).size())

# --- Parcelas corregidas manualmente (de la v1) ---
plot_fix = {
    "p28-8878": 28, "p60-8885": 60, "p48-3698": 48, "p68-8878": 68, "p68-8885": 68,
    "p10-252": 10, "p12-304": 12, "p13-315": 13,
    "p09-230": 9,      # decía 01
    "p62-8669": 48,    # prefijo no coincide pero verificado manualmente
    "p28-8974": 23,    # idem
    "p12-221": 12,     # Sumaco, sin Plot
}
for sample, plot in plot_fix.items():
    all_df.loc[all_df['Sample_ID'] == sample, 'Plot'] = plot

# --- Columnas intercambiadas en Oyacachi: [C_b]% <-> δ15N (de la v1) ---
mask_swap = ((all_df['Sample_ID'] == 'p82-4044') & (all_df['Year'] == 2011)) | (all_df['Sample_ID'] == 'p81-8023')
print('Filas con swap C_bulk/δ15N:', mask_swap.sum())
all_df.loc[mask_swap, ['C_bulk_pct', 'd15N_bulk']] = all_df.loc[mask_swap, ['d15N_bulk', 'C_bulk_pct']].to_numpy()
# (antes del swap, el valor negativo de C_bulk no se convirtió porque |x|>=1, y el δ15N ~43 ya era %; queda correcto)

# --- Type vacío: jh- = herbario, resto = pulverizado ---
m = all_df['Type'].isna()
all_df.loc[m & all_df['Sample_ID'].str.startswith('jh-', na=False), 'Type'] = 'leaf herbarium'
all_df.loc[all_df['Type'].isna(), 'Type'] = 'leaf pulverized'
all_df['Year_imputado'] = all_df['Year_imputado'].fillna(False).astype(bool)

# --- Errores de tipeo en género (para que el mismo árbol no parezca de géneros distintos) ---
GENUS_FIX = {'Elaegia': 'Elaeagia', 'Farameo': 'Faramea', 'Hyeronima': 'Hieronyma',
             'Weinmania': 'Weinmannia', 'Tetrochidium': 'Tetrorchidium'}
all_df['genus'] = all_df['genus'].astype('string').str.strip().replace(GENUS_FIX)

Filas totales: 746
Project  Site      
NAPO     Galeras       268
         Guacamayos     89
         Oyacachi       60
         Selva Viva    148
         Sumaco         93
NUMEX    BoNu           26
         CaNu           20
         SFNu           41
         NaN             1
dtype: int64
Filas con swap C_bulk/δ15N: 2


## 6. Elevación

In [10]:
ELEV_NUMEX = {'BoNu': 1000, 'SFNu': 2000, 'CaNu': 3000}   # del propio NUMEXC.xlsx (las filas 2009/2011 venían vacías)

def get_elev(row):
    if row['Site'] in ELEV_NUMEX:
        return ELEV_NUMEX[row['Site']]
    if pd.notna(row.get('Elevation_m')):
        return row['Elevation_m']
    if pd.isna(row['Plot']):
        return np.nan
    return elevation_dict.get((row['Site'], int(row['Plot'])), np.nan)

all_df['Elevation_m'] = all_df.apply(get_elev, axis=1)
print('Sin elevación:', all_df['Elevation_m'].isna().sum())

Sin elevación: 1


## 7. Llave única de árbol (`Tree_key`)

El mismo árbol aparece con su **ID viejo** (`old_treeID`, censo 2006) y/o su **ID nuevo** (`new_TreeID`, re-etiquetado). Para juntar todas sus muestras:

1. Con las filas de remuestreo (Year ≥ 2023, que traen ambos IDs verificados en campo) se arma una tabla `nuevo → viejo` por sitio.
2. Cada fila usa su ID viejo; si no lo tiene, traduce el nuevo con esa tabla; si tampoco hay, usa el nuevo tal cual.
3. Si no hay ningún ID, se usa el número del `Sample_ID` (excepto en `jh-`, donde ese número es del herbario).

`Tree_key = Sitio | O<id viejo>` o `Sitio | N<id nuevo>`. En NUMEXC: `Sitio | p<Plot> | T<TreeID>`.

Se usa solo el sitio (no la parcela) en NAPO porque hay errores de parcela conocidos y los tags son únicos por sitio.

In [11]:
napo = all_df['Project'] == 'NAPO'
rec = all_df[napo & (all_df['Year'] >= 2023) & all_df['old_treeID'].notna() & all_df['new_TreeID'].notna()]
new2old = {}
conflictos_new = []
for (site, new), grp in rec.groupby(['Site', 'new_TreeID']):
    olds = sorted(grp['old_treeID'].unique())
    new2old[(site, new)] = olds[0]
    if len(olds) > 1:
        conflictos_new.append((site, new, olds))
print('Pares nuevo→viejo:', len(new2old), '| conflictos:', conflictos_new)

def tree_key(r):
    if r['Project'] == 'NUMEX':
        if pd.isna(r['Site']) or pd.isna(r['new_TreeID']):
            return pd.NA, 'sin ID'
        return f"{r['Site']}|p{int(r['Plot'])}|T{int(r['new_TreeID'])}", 'TreeID'
    site = r['Site']
    if pd.notna(r['old_treeID']):
        return f"{site}|O{int(r['old_treeID'])}", 'old_treeID'
    if pd.notna(r['new_TreeID']):
        old = new2old.get((site, r['new_TreeID']))
        if old is not None:
            return f"{site}|O{int(old)}", 'new_TreeID→old'
        return f"{site}|N{int(r['new_TreeID'])}", 'new_TreeID'
    n = id_number(r['Sample_ID'])
    if pd.notna(n) and not str(r['Sample_ID']).startswith('jh-'):
        old = new2old.get((site, n))
        if old is not None:
            return f"{site}|O{int(old)}", 'Sample_ID→old'
        return f"{site}|N{int(n)}", 'Sample_ID'
    return pd.NA, 'sin ID'

all_df[['Tree_key', 'Tree_key_fuente']] = all_df.apply(lambda r: pd.Series(tree_key(r)), axis=1)
all_df['Tree_key_fuente'].value_counts()

Pares nuevo→viejo: 214 | conflictos: []


Tree_key_fuente
old_treeID        619
TreeID             87
new_TreeID         35
sin ID              3
new_TreeID→old      1
Sample_ID           1
Name: count, dtype: int64

## 8. Control de calidad (`QC_flag`)

Rangos esperados (después de homogeneizar): C celulosa 35–55 %, C bulk 35–60 %, N 0.3–6 %, δ13C −40 a −20 ‰, δ15N −10 a +15 ‰.

In [12]:
all_df['QC_flag'] = ''
def flag(mask, msg):
    all_df.loc[mask, 'QC_flag'] += msg + '; '

RANGOS = {'C_cel_pct': (35, 55), 'C_bulk_pct': (35, 60), 'N_bulk_pct': (0.3, 6),
          'd13C_cel': (-40, -20), 'd13C_bulk': (-40, -20), 'd15N_bulk': (-10, 15), 'O_pct': (35, 60)}
for c, (lo, hi) in RANGOS.items():
    flag(all_df[c].notna() & ~all_df[c].between(lo, hi), f'{c} fuera de rango [{lo},{hi}]')

flag(all_df['Plot'].isna(), 'Plot vacío')
flag(all_df['Site'].isna(), 'Site vacío')
flag(all_df['Year'].isna(), 'Year vacío')
flag(all_df['Elevation_m'].isna() & all_df['Plot'].notna(), 'Sin elevación en elevation_dict')
flag(all_df['Tree_key'].isna(), 'Sin ID de árbol')

all_df['Plot_from_ID'] = all_df['Sample_ID'].apply(implied_plot).astype('Int64')
mm = all_df['Plot_from_ID'].notna() & all_df['Plot'].notna() & (all_df['Plot_from_ID'] != all_df['Plot'])
ya_verificados = all_df['Sample_ID'].isin(['p62-8669', 'p28-8974'])
flag(mm & ~ya_verificados, 'Prefijo de parcela en Sample_ID ≠ Plot')

# Remuestreos: el número del Sample_ID debería ser el new_TreeID
num = all_df['Sample_ID'].apply(id_number).astype('Int64')
m = napo & (all_df['Year'] >= 2023) & all_df['new_TreeID'].notna() & num.notna() & (num != all_df['new_TreeID'])
flag(m, 'Número del Sample_ID ≠ new_TreeID')

# Un ID viejo con dos IDs nuevos distintos dentro del mismo sitio
pares = all_df[napo & all_df['old_treeID'].notna() & all_df['new_TreeID'].notna()]
multi = pares.groupby(['Site', 'old_treeID'])['new_TreeID'].nunique()
multi = multi[multi > 1].reset_index()[['Site', 'old_treeID']]
m = all_df.set_index(['Site', 'old_treeID']).index.isin(multi.set_index(['Site', 'old_treeID']).index)
flag(m, 'old_treeID con 2+ new_TreeID distintos')

# new_TreeID que en realidad es un número de herbario del mismo sitio (ej. p48-3698)
herb = all_df[napo].assign(h=all_df['JH herbarium collections'].astype('string').str.extract(r'^(\d+)')[0])
herb_set = set(zip(herb['Site'], pd.to_numeric(herb['h'], errors='coerce')))
m = napo & all_df['old_treeID'].isna() & all_df['new_TreeID'].notna() & all_df.apply(lambda r: (r['Site'], r['new_TreeID']) in herb_set, axis=1)
flag(m, 'new_TreeID coincide con un nº de herbario (¿ID equivocado?)')

# Réplicas de laboratorio: misma muestra/año con δ13C celulosa que difiere > 0.5 ‰
rep = all_df.groupby(['Site', 'Sample_ID', 'Year'], dropna=False)['d13C_cel'].agg(['count', lambda s: s.max() - s.min()])
rep.columns = ['n', 'rango']
mala = rep[(rep['n'] > 1) & (rep['rango'] > 0.5)].index
m = all_df.set_index(['Site', 'Sample_ID', 'Year']).index.isin(mala)
flag(m, 'Réplicas con δ13C_cel que difiere > 0.5 ‰')

# Género con números (parece un reemplazo masivo mal hecho: 'Rubi28', 'Sapo28', 'Apo28')
flag(all_df['genus'].str.contains(r'\d', na=False), 'Género con números (¿reemplazo mal hecho?)')

all_df['n_replicas'] = all_df.groupby(['Site', 'Sample_ID', 'Year'], dropna=False)['Sample_ID'].transform('size')
all_df['QC_flag'] = all_df['QC_flag'].str.rstrip('; ')
print('Filas marcadas:', (all_df['QC_flag'] != '').sum(), 'de', len(all_df))
all_df.loc[all_df['QC_flag'] != '', 'QC_flag'].str.split('; ').explode().value_counts()

Filas marcadas: 60 de 746


QC_flag
Réplicas con δ13C_cel que difiere > 0.5 ‰                      24
C_cel_pct fuera de rango [35,55]                               14
C_bulk_pct fuera de rango [35,60]                               7
Género con números (¿reemplazo mal hecho?)                      6
Sin ID de árbol                                                 3
Prefijo de parcela en Sample_ID ≠ Plot                          2
Número del Sample_ID ≠ new_TreeID                               2
old_treeID con 2+ new_TreeID distintos                          2
Year vacío                                                      2
new_TreeID coincide con un nº de herbario (¿ID equivocado?)     1
Plot vacío                                                      1
Site vacío                                                      1
Name: count, dtype: int64

## 9. Pendientes por medir

Se evalúa **por muestra** (`Site + Sample_ID + Year`): si una réplica tiene el dato, la muestra no cuenta como pendiente.
- **Celulosa**: `C_cel_pct`, `d13C_cel`
- **Bulk**: `C_bulk_pct`, `d13C_bulk`, `N_bulk_pct`, `d15N_bulk`

In [13]:
first = lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA
muestras = (all_df.groupby(['Project', 'Site', 'Sample_ID', 'Year'], dropna=False)
            .agg(Plot=('Plot', first), Type=('Type', first), Tree_key=('Tree_key', first),
                 genus=('genus', first), species=('species', first), n_replicas=('Sample_ID', 'size'),
                 **{c: (c, 'mean') for c in CORE_ISO})
            .reset_index())

def estado(r):
    falta = [c for c in CORE_ISO if pd.isna(r[c])]
    cel = any(c in falta for c in CEL)
    blk = any(c in falta for c in BULK)
    if not falta:
        return 'completo', ''
    if cel and blk:
        e = 'falta celulosa y bulk'
    elif cel:
        e = 'falta celulosa'
    else:
        e = 'falta bulk'
    return e, ', '.join(falta)

muestras[['Estado', 'Variables_faltantes']] = muestras.apply(lambda r: pd.Series(estado(r)), axis=1)
resumen_pend = pd.crosstab([muestras['Site'], muestras['Year']], muestras['Estado'], margins=True, margins_name='Total')
resumen_pend

Estado           completo  falta bulk  falta celulosa  falta celulosa y bulk  Total
Site       Year                                                                    
BoNu       2009        10           0               3                      0     13
           2024        12           0               0                      0     12
CaNu       2009         7           0               3                      0     10
           2024        10           0               0                      0     10
Galeras    2006        97           7              49                      4    157
           2025        77           5               1                      0     83
Guacamayos 2005         4           0               0                      0      4
           2006        37          12               0                      0     49
           2007         2           0               0                      0      2
           2023        28           0               0                      0     28
Oyacachi   2009         1           0               0                      0      1
           2011        24           0               2                      0     26
           2023        30           0               3                      0     33
SFNu       2011        13           0               1                      0     14
           2023        25           0               0                      0     25
           2024         2           0               0                      0      2
Selva Viva 2006        49          12               6                      3     70
           2025        54           4               2                      0     60
Sumaco     2006        42           0               2                      1     45
           2025        44           1               1                      0     46
Total                 568          41              73                      8    690

## 10. Coincidencias de árboles entre años

Para cada árbol (`Tree_key`): en qué años tiene muestra, cuántos años separan la primera de la última, y si en cada año ya está medida la celulosa (δ13C) y el bulk.
Además se marca si el **género cambia** entre años (posible error de identidad del árbol o de determinación).

In [14]:
# Sinónimos / cambios nomenclaturales: no cuentan como 'género cambia'
SINONIMOS = {'Lucuma': 'Pouteria', 'Jupunba': 'Abarema', 'Lacunaria': 'Quiina'}

def norm_genus(s):
    s = s.dropna().astype(str).str.strip().str.capitalize()
    s = s[~s.isin(['Indeterminado']) & ~s.str.contains(r'\d')]
    return sorted(s.replace(SINONIMOS).unique())

filas = []
for (proj, site, tk), grp in muestras[muestras['Tree_key'].notna() & muestras['Year'].notna()].groupby(['Project', 'Site', 'Tree_key']):
    years = sorted(grp['Year'].unique())
    by_year = grp.groupby('Year')
    cel = {y: by_year.get_group(y)['d13C_cel'].notna().any() for y in years}
    blk = {y: by_year.get_group(y)['d13C_bulk'].notna().any() for y in years}
    gen = norm_genus(grp['genus'])
    filas.append({
        'Project': proj, 'Site': site, 'Tree_key': tk,
        'Plot': first(grp['Plot']),
        'old_treeID': first(all_df.loc[all_df['Tree_key'] == tk, 'old_treeID']),
        'new_TreeID': first(all_df.loc[(all_df['Tree_key'] == tk) & all_df['new_TreeID'].notna(), 'new_TreeID']),
        'genus': ' / '.join(gen), 'species': first(grp['species']),
        'n_años': len(years),
        'Años': ', '.join(str(int(y)) for y in years),
        'Primer_año': int(years[0]), 'Último_año': int(years[-1]),
        'Dif_años': int(years[-1] - years[0]),
        'Sample_IDs': ', '.join(sorted(grp['Sample_ID'].dropna().astype(str).unique())),
        'Años_con_celulosa': ', '.join(str(int(y)) for y in years if cel[y]),
        'Años_sin_celulosa': ', '.join(str(int(y)) for y in years if not cel[y]),
        'Años_sin_bulk': ', '.join(str(int(y)) for y in years if not blk[y]),
        'Par_celulosa_completo': len(years) > 1 and sum(cel.values()) >= 2,
        'Género_cambia': len(gen) > 1,
    })
arboles = pd.DataFrame(filas).sort_values(['Project', 'Site', 'n_años', 'Tree_key'], ascending=[True, True, False, True])

multi = arboles[arboles['n_años'] > 1]
resumen_sitio = arboles.groupby(['Project', 'Site']).agg(
    Árboles_total=('Tree_key', 'size'),
    Árboles_con_2mas_años=('n_años', lambda s: (s > 1).sum()),
    Pares_celulosa_completos=('Par_celulosa_completo', 'sum'),
).join(multi.groupby(['Project', 'Site']).agg(
    Combinaciones_de_años=('Años', lambda s: '; '.join(f'{k} (n={v})' for k, v in s.value_counts().items())),
    Dif_años_min=('Dif_años', 'min'), Dif_años_max=('Dif_años', 'max'),
    Género_cambia=('Género_cambia', 'sum'),
)).reset_index()
resumen_sitio['Pares_por_completar'] = resumen_sitio['Árboles_con_2mas_años'] - resumen_sitio['Pares_celulosa_completos']
resumen_sitio

,Project,Site,Árboles_total,Árboles_con_2mas_años,Pares_celulosa_completos,Combinaciones_de_años,Dif_años_min,Dif_años_max,Género_cambia,Pares_por_completar
0,NAPO,Galeras,157,58,52,"2006, 2025 (n=58)",19,19,0,6
1,NAPO,Guacamayos,55,22,22,"2006, 2023 (n=16); 2005, 2006, 2023 (n=4); 200...",17,18,1,0
2,NAPO,Oyacachi,33,26,21,"2011, 2023 (n=25); 2009, 2011, 2023 (n=1)",12,14,0,5
3,NAPO,Selva Viva,85,41,37,"2006, 2025 (n=41)",19,19,0,4
4,NAPO,Sumaco,75,16,16,"2006, 2025 (n=16)",19,19,0,0
5,NUMEX,BoNu,14,11,8,"2009, 2024 (n=11)",15,15,0,3
6,NUMEX,CaNu,11,9,6,"2009, 2024 (n=9)",15,15,0,3
7,NUMEX,SFNu,35,6,6,"2011, 2023 (n=6)",12,12,0,0


In [15]:
# Tabla árbol x año (cuántas muestras), útil para ver el diseño de un vistazo
combos = (multi.groupby(['Site', 'Años', 'Dif_años']).size().rename('n_árboles').reset_index()
          .sort_values(['Site', 'n_árboles'], ascending=[True, False]))
combos

,Site,Años,Dif_años,n_árboles
0,BoNu,"2009, 2024",15,11
1,CaNu,"2009, 2024",15,9
2,Galeras,"2006, 2025",19,58
5,Guacamayos,"2006, 2023",17,16
3,Guacamayos,"2005, 2006, 2023",18,4
4,Guacamayos,"2006, 2007, 2023",17,2
7,Oyacachi,"2011, 2023",12,25
6,Oyacachi,"2009, 2011, 2023",14,1
8,SFNu,"2011, 2023",12,6
9,Selva Viva,"2006, 2025",19,41


## 11. Guardar Excel con celdas vacías resaltadas

Hojas:
- `Resumen_sitio` – coincidencias de árboles por sitio.
- `Coincidencias_años` – combinaciones de años y diferencia en años.
- `Arboles` – una fila por árbol (verde = par de celulosa completo; rojo = el género cambia).
- `Pendientes_resumen` – conteo de muestras completas / pendientes por sitio y año.
- `Pendientes_muestras` – una fila por muestra; **celdas vacías resaltadas en rojo**.
- `Datos_concatenados` – todas las filas; **celdas de isótopos vacías resaltadas en rojo**, filas con QC en amarillo (columna `QC_flag`).
- `Errores_a_revisar` – solo filas con `QC_flag`.
- `Conversion_unidades`, `Diccionario_columnas`, `NUMEXC_rasgos`.

In [16]:
front = ['Project', 'Site', 'Plot', 'Subplot', 'Elevation_m', 'Tree_key', 'old_treeID', 'new_TreeID',
         'Sample_ID', 'Year', 'Year_imputado', 'Type', 'n_replicas', 'family', 'genus', 'species'] + CORE_ISO + O_ISO + ['QC_flag']
drop_cols = ['Plot_from_ID', 'comment']
otras = [c for c in all_df.columns if c not in front and c not in drop_cols]
out = all_df[[c for c in front if c in all_df.columns] + otras].sort_values(['Project', 'Site', 'Plot', 'Tree_key', 'Year'])
errores = out[out['QC_flag'] != '']

dicc = pd.DataFrame([
    ('C_cel_pct', '% C en celulosa', '%', '[C]%'),
    ('d13C_cel', 'δ13C celulosa', '‰ VPDB', 'δ13C (‰ v.s.V-PDB)'),
    ('C_bulk_pct', '% C hoja entera (bulk)', '%', '[C_b]% ; Sumaco: [C]'),
    ('d13C_bulk', 'δ13C hoja entera (bulk)', '‰ VPDB', 'bulk_δ13C (‰ v.s.V-PDB)'),
    ('N_bulk_pct', '% N hoja entera (bulk)', '%', '[N]'),
    ('d15N_bulk', 'δ15N hoja entera (bulk)', '‰ AIR', 'δ15N (‰ v.s. V-PDB) ; Sumaco: δ15N (‰ v.s.AIR)'),
    ('O_pct', '% O (solo NUMEXC)', '%', '[O]'),
    ('d18O', 'δ18O (solo NUMEXC)', '‰ VSMOW', 'δ18O in ‰ vs DeltaSMOW'),
    ('Tree_key', 'Llave única de árbol (une ID viejo y nuevo)', '', 'calculada'),
    ('Tree_key_fuente', 'De dónde salió la llave', '', 'calculada'),
    ('n_replicas', 'Nº de filas (réplicas) de la misma muestra y año', '', 'calculada'),
    ('Year_imputado', 'True si el año se imputó (Galeras jh- → 2006; tag 89xx → 2025)', '', 'calculada'),
    ('source_file / source_row', 'Archivo y fila de origen, para volver al dato crudo', '', 'calculada'),
], columns=['Columna', 'Descripción', 'Unidad', 'Nombre original'])

nx_traits = nx_raw.copy()
nx_traits.insert(0, 'Tree_key', nx_traits.apply(
    lambda r: f"{r['Site']}|p{int(r['Plot'])}|T{int(r['TreeID'])}" if pd.notna(r['Site']) and pd.notna(r['TreeID']) else pd.NA, axis=1))
nx_traits = nx_traits.drop(columns=[c for c in nx_traits.columns if str(c).startswith('Unnamed')])

out_path = BASE_DIR / 'isotopos_concatenado.xlsx'
hojas = {
    'Resumen_sitio': resumen_sitio,
    'Coincidencias_años': combos,
    'Arboles': arboles,
    'Pendientes_resumen': resumen_pend.reset_index(),
    'Pendientes_muestras': muestras.sort_values(['Project', 'Site', 'Year', 'Sample_ID']),
    'Datos_concatenados': out,
    'Errores_a_revisar': errores,
    'Conversion_unidades': pd.DataFrame(conversiones),
    'Diccionario_columnas': dicc,
    'NUMEXC_rasgos': nx_traits,
}
with pd.ExcelWriter(out_path, engine='openpyxl') as w:
    for name, df in hojas.items():
        df.to_excel(w, sheet_name=name, index=False)
print('Guardado:', out_path)

Guardado: /home/claude/data/isotopos_concatenado.xlsx


In [17]:
# Formato: celdas vacías en rojo, encabezados, anchos, filtros
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

ROJO = PatternFill('solid', start_color='F8CBAD')
AMARILLO = PatternFill('solid', start_color='FFF2CC')
VERDE = PatternFill('solid', start_color='C6EFCE')
HEAD = PatternFill('solid', start_color='1F4E78')

wb = load_workbook(out_path)

def formato_basico(ws):
    for c in ws[1]:
        c.fill = HEAD
        c.font = Font(bold=True, color='FFFFFF')
        c.alignment = Alignment(wrap_text=True, vertical='center')
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    for i, col in enumerate(ws.iter_cols(min_row=1, max_row=min(ws.max_row, 200)), 1):
        largo = max(len(str(c.value)) if c.value is not None else 0 for c in col)
        ws.column_dimensions[get_column_letter(i)].width = min(max(largo + 2, 8), 45)

def resaltar_vacias(ws, columnas):
    idx = {c.value: c.column for c in ws[1]}
    for nombre in columnas:
        if nombre not in idx:
            continue
        col = idx[nombre]
        for r in range(2, ws.max_row + 1):
            if ws.cell(r, col).value in (None, ''):
                ws.cell(r, col).fill = ROJO

for ws in wb.worksheets:
    formato_basico(ws)

ws = wb['Datos_concatenados']
resaltar_vacias(ws, CORE_ISO)
qc_col = [c.column for c in ws[1] if c.value == 'QC_flag'][0]
for r in range(2, ws.max_row + 1):
    if ws.cell(r, qc_col).value:
        ws.cell(r, qc_col).fill = AMARILLO

ws = wb['Pendientes_muestras']
resaltar_vacias(ws, CORE_ISO)
est_col = [c.column for c in ws[1] if c.value == 'Estado'][0]
for r in range(2, ws.max_row + 1):
    ws.cell(r, est_col).fill = VERDE if ws.cell(r, est_col).value == 'completo' else ROJO

ws = wb['Arboles']
hdr = {c.value: c.column for c in ws[1]}
for r in range(2, ws.max_row + 1):
    if ws.cell(r, hdr['Par_celulosa_completo']).value:
        ws.cell(r, hdr['Par_celulosa_completo']).fill = VERDE
    if ws.cell(r, hdr['Género_cambia']).value:
        ws.cell(r, hdr['Género_cambia']).fill = ROJO
    if ws.cell(r, hdr['Años_sin_celulosa']).value:
        ws.cell(r, hdr['Años_sin_celulosa']).fill = ROJO

wb.save(out_path)
print('Formato aplicado:', out_path)

Formato aplicado: /home/claude/data/isotopos_concatenado.xlsx


## 12. Resumen de cosas a revisar manualmente
(se imprime abajo; los detalles están en la hoja `Errores_a_revisar`)

In [18]:
print('=== Muestras por estado ===')
print(muestras['Estado'].value_counts().to_string())
print('\n=== Motivos de QC ===')
print(errores['QC_flag'].str.split('; ').explode().value_counts().to_string())
print('\n=== Árboles cuyo género cambia entre años ===')
print(arboles.loc[arboles['Género_cambia'], ['Site', 'Tree_key', 'genus', 'Años', 'Sample_IDs']].to_string(index=False))

=== Muestras por estado ===
Estado
completo                 568
falta celulosa            74
falta bulk                42
falta celulosa y bulk      8

=== Motivos de QC ===
QC_flag
Réplicas con δ13C_cel que difiere > 0.5 ‰                      24
C_cel_pct fuera de rango [35,55]                               14
C_bulk_pct fuera de rango [35,60]                               7
Género con números (¿reemplazo mal hecho?)                      6
Sin ID de árbol                                                 3
Prefijo de parcela en Sample_ID ≠ Plot                          2
Número del Sample_ID ≠ new_TreeID                               2
old_treeID con 2+ new_TreeID distintos                          2
Year vacío                                                      2
new_TreeID coincide con un nº de herbario (¿ID equivocado?)     1
Plot vacío                                                      1
Site vacío                                                      1

=== Árboles cuyo género c